# Foul checker — feed the model any image (Colab)
**Team: Sahayu R · Rishi D · Shashank R · Brian L · Abhinav K · Ishmeet S**

Upload any picture (or paste an image URL) and your fine-tuned RF-DETR tells you
what it sees: draws boxes and prints a plain-language verdict. Inference runs on
Roboflow's servers — no GPU runtime needed, nothing is trained or changed.

**Two-stage judge:** your RF-DETR detector finds *who* is involved; a second
classifier (ResNet-18, trained on clean-vs-foul tackle images) rules *whether*
the tackle is clean (ball won first) or a foul. When they disagree, the verdict
is flagged for human review instead of guessing.

Verified (2026-07-18): balanced accuracy 92.8% across four held-out sets;
~83–90% on the classifier's own domain is the conservative estimate. On known
clean tackles the detector alone said foul every time — the classifier
correctly overruled it.

A third model screens for **handball** (92% balanced on small test sets).
Its training data has NO goalkeeper labels, so it cannot apply the keeper
exemption — every handball flag is marked needs-human-review by design.

Not a general image classifier: upload a cat and the honest answer is
"nothing recognized".

*Clean/foul training data includes the VarCNN dataset — credit: Aamir Ahmad
Ansari, github.com/aamir09/VarCnn.*

## 1. Setup

In [ ]:
import os, getpass, sys
!mkdir -p /content/ai-referee/src/airef
%cd /content/ai-referee
!pip -q install opencv-python-headless matplotlib numpy av ultralytics

sys.path.insert(0, 'src')
os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow PRIVATE API key: ')
print('key set for this session only — not written to disk')

## 2. Project files (tested code)

In [ ]:
%%writefile src/airef/__init__.py
"""AI Referee: automated soccer violation detection from video clips."""

__version__ = "0.1.0"


In [ ]:
%%writefile src/airef/device.py
"""Device selection: CUDA (Colab) / MPS (Apple Silicon) / CPU.

Kept in one place so training, evaluation and inference agree. Apple's MPS
backend has no GradScaler and no stable fp16 autocast, so mixed precision is
enabled for CUDA only — on MPS we train in fp32, which is what the M-series
unified memory wants anyway.
"""
from __future__ import annotations

import os

import torch


def pick_device(prefer: str | None = None) -> str:
    """Return 'cuda', 'mps' or 'cpu'.

    Override with the AIREF_DEVICE env var (e.g. AIREF_DEVICE=cpu) when the
    MPS path misbehaves on a particular op.
    """
    choice = prefer or os.environ.get("AIREF_DEVICE")
    if choice:
        return choice
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def amp_enabled(device: str) -> bool:
    """Mixed precision is CUDA-only here (see module docstring)."""
    return device == "cuda"


def pin_memory(device: str) -> bool:
    """pin_memory only means anything for CUDA host->device copies."""
    return device == "cuda"


def dataloader_workers(device: str) -> int:
    """macOS spawns (not forks) workers; 2 is fine and keeps decode off the main thread."""
    return 2


In [ ]:
%%writefile src/airef/roboflow_client.py
"""Binary foul / no-foul detection via the team's Roboflow workflow (REST).

Why REST and not the official `inference-sdk`: every published inference-sdk
release requires Python < 3.13 and this project runs 3.13, so the SDK cannot
be installed. The serverless REST endpoint is Roboflow's documented fallback
and needs no extra dependencies.

Why the child workflow and not the "...Logic" wrapper: the wrapper
(football-foul-detection-vfootball-foul-detection-u1r8g-1-rfdetr-medium-t1-logic)
is broken server-side — its inner_workflow step binds a `model_id` parameter
that the child workflow does not declare, so every run returns HTTP 500
(InnerWorkflowParameterBindingsUnknownInputError). The child workflow
`football-foul-detection-u1r8g` is the actual model run and works. Fix the
wrapper in the Roboflow UI and you can pass its slug via `workflow_id=`.

The API key comes from the ROBOFLOW_API_KEY environment variable, or from a
git-ignored `.env` file at the repo root. Never hardcode or commit it.

Grounded response shape (captured from a real run, 2026-07-18 — see
runs/roboflow_ground/):

    {"outputs": [{"predictions": {
        "image": {"width": W, "height": H},
        "predictions": [{"x":..,"y":..,"width":..,"height":..,
                         "confidence":..,"class":"fouling_player"|"victim"|"foul",
                         "class_id":..,"detection_id":..,"parent_id":..}, ...]}}],
     "profiler_trace": []}
"""
from __future__ import annotations

import base64
import json
import os
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

API_URL = "https://serverless.roboflow.com"
WORKSPACE = "ishmeet-singh-csaoh"
DEFAULT_WORKFLOW = "football-foul-detection-u1r8g"

# Soccer-specific ball detector (RF-DETR-small). The "...-logic" wrapper works
# for this one, but we target the child workflow for consistency with the foul
# client and to avoid the inner_workflow binding class of bug.
BALL_WORKFLOW = "soccer-ball-detection-2-teahj"

# Detection classes that signal a foul event (the model also emits "victim").
FOUL_CLASSES = frozenset({"foul", "fouling_player"})


class RoboflowError(RuntimeError):
    """Request or response-shape failure talking to Roboflow."""


class RoboflowAuthError(RoboflowError):
    """Bad or missing API key."""


@dataclass
class Detection:
    cls: str
    confidence: float
    x: float
    y: float
    width: float
    height: float


@dataclass
class FoulCall:
    """Binary verdict for one image.

    confidence is a heuristic: the strongest foul-class detection when foul,
    else 1 - the strongest sub-threshold foul-class detection (1.0 if none).
    """
    foul: bool
    confidence: float
    detections: list[Detection] = field(default_factory=list)
    image_wh: tuple[int, int] = (0, 0)


def _api_key() -> str:
    key = os.environ.get("ROBOFLOW_API_KEY")
    if not key:
        env_file = Path(__file__).resolve().parents[2] / ".env"
        if env_file.exists():
            for line in env_file.read_text().splitlines():
                name, _, value = line.strip().partition("=")
                if name == "ROBOFLOW_API_KEY" and value:
                    key = value.strip().strip("'\"")
                    break
    if not key:
        raise RoboflowAuthError(
            "ROBOFLOW_API_KEY not set (env var or .env at the repo root). "
            "Get it at app.roboflow.com/settings/api."
        )
    return key


def _to_base64(image: str | Path | bytes) -> str:
    data = image if isinstance(image, bytes) else Path(image).read_bytes()
    return base64.b64encode(data).decode()


def run_foul_workflow(
    image: str | Path | bytes,
    workflow_id: str = DEFAULT_WORKFLOW,
    timeout: float = 60.0,
    retries: int = 2,
) -> tuple[list[Detection], tuple[int, int]]:
    """Run one image through the workflow; return (detections, (width, height)).

    Retries transient failures (network errors, HTTP 5xx) with exponential
    backoff. 4xx errors don't retry; 401/403 raise RoboflowAuthError.
    """
    body = json.dumps({
        "api_key": _api_key(),
        "inputs": {"image": {"type": "base64", "value": _to_base64(image)}},
    }).encode()
    url = f"{API_URL}/{WORKSPACE}/workflows/{workflow_id}"

    last_err: Exception | None = None
    for attempt in range(retries + 1):
        try:
            req = urllib.request.Request(
                url, data=body, headers={"Content-Type": "application/json"}
            )
            with urllib.request.urlopen(req, timeout=timeout) as r:
                resp = json.load(r)
            break
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:300]
            if e.code in (401, 403):
                raise RoboflowAuthError(f"HTTP {e.code}: {detail}") from e
            if 400 <= e.code < 500:
                raise RoboflowError(f"HTTP {e.code}: {detail}") from e
            last_err = RoboflowError(f"HTTP {e.code}: {detail}")
        except (urllib.error.URLError, TimeoutError, json.JSONDecodeError) as e:
            last_err = RoboflowError(f"request failed: {e}")
        if attempt < retries:
            time.sleep(0.5 * 2 ** attempt)
    else:
        raise last_err  # type: ignore[misc]

    # Defensive parse against the grounded shape; never assume, never keep blobs.
    try:
        block = resp["outputs"][0]["predictions"]
        # width/height come back as null (not absent) when the model finds
        # nothing in the image, so `or {}` / `or 0` — not .get defaults.
        img = block.get("image") or {}
        raw = block["predictions"]
    except (KeyError, IndexError, TypeError) as e:
        keys = list(resp)[:8] if isinstance(resp, dict) else type(resp).__name__
        raise RoboflowError(f"unexpected response shape (top-level: {keys})") from e

    dets = [
        Detection(
            cls=str(p.get("class", "?")),
            confidence=float(p.get("confidence", 0.0)),
            x=float(p.get("x", 0.0)), y=float(p.get("y", 0.0)),
            width=float(p.get("width", 0.0)), height=float(p.get("height", 0.0)),
        )
        for p in raw
    ]
    return dets, (int(img.get("width") or 0), int(img.get("height") or 0))


@dataclass
class BallDetection:
    confidence: float
    x: float
    y: float
    width: float
    height: float


def detect_ball(
    image: str | Path | bytes,
    threshold: float = 0.4,
    workflow_id: str = BALL_WORKFLOW,
    timeout: float = 60.0,
    retries: int = 2,
) -> BallDetection | None:
    """Locate the soccer ball with the team's RF-DETR ball model.

    Returns the highest-confidence ball at/above `threshold`, or None. Grounded
    on the real response (2026-07-20): same envelope as the foul workflow —
    outputs[0].predictions.predictions[], with image width/height null when
    nothing is found.

    Replaces the generic COCO detector, which hallucinated balls below 0.2 and
    could not see a ball pressed against a body (measured: 0.13 on a real
    handball). This model scores that same case at 0.51.
    """
    dets, _ = run_foul_workflow(image, workflow_id=workflow_id,
                                timeout=timeout, retries=retries)
    balls = [d for d in dets if d.confidence >= threshold]
    if not balls:
        return None
    b = max(balls, key=lambda d: d.confidence)
    return BallDetection(b.confidence, b.x, b.y, b.width, b.height)


def classify_foul(
    image: str | Path | bytes,
    threshold: float = 0.5,
    workflow_id: str = DEFAULT_WORKFLOW,
) -> FoulCall:
    """Binary foul / no-foul call for one image.

    Rule: FOUL iff any detection of a class in FOUL_CLASSES has
    confidence >= threshold. Tune the threshold on labeled frames (see
    ai_referee_roboflow_binary.ipynb) before trusting the verdicts.
    """
    dets, wh = run_foul_workflow(image, workflow_id=workflow_id)
    foul_scores = [d.confidence for d in dets if d.cls in FOUL_CLASSES]
    hits = [s for s in foul_scores if s >= threshold]
    if hits:
        return FoulCall(True, max(hits), dets, wh)
    return FoulCall(False, 1.0 - max(foul_scores, default=0.0), dets, wh)


In [ ]:
%%writefile src/airef/tackle_classifier.py
"""Second-stage clean/foul tackle classifier for the foul checker.

ResNet-18 fine-tuned (locally, 2026-07-18) on a domain-mixed set:
- VarCNN clean/foul tackle crops — credit: Aamir Ahmad Ansari,
  github.com/aamir09/VarCnn (free use with attribution)
- the team's own Roboflow foul images (train split only)
- locally-held clean-challenge frames

Verified on four held-out sets: balanced accuracy 92.8% overall;
~83-90% within VarCNN, where both classes share one camera style (the more
conservative and more honest estimate — see session notes).

The detector (roboflow_client) answers WHO is involved; this model answers
WHETHER the tackle is clean (ball won first) or a foul.
"""
from __future__ import annotations

import io
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

from .device import pick_device

DEFAULT_WEIGHTS = Path(__file__).resolve().parents[2] / "runs/clean_tackle/resnet18_mixed.pt"

_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_model: nn.Module | None = None
_classes: list[str] = []
_device = "cpu"


def load_classifier(weights: str | Path = DEFAULT_WEIGHTS) -> None:
    """Load weights once; later classify_tackle calls reuse them."""
    global _model, _classes, _device
    weights = Path(weights)
    if not weights.exists():
        raise FileNotFoundError(
            f"classifier weights not found at {weights} — "
            "copy runs/clean_tackle/resnet18_mixed.pt there first."
        )
    _device = pick_device()
    ck = torch.load(weights, map_location=_device, weights_only=False)
    classes = list(ck["classes"])
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, len(classes))   # 2- or 3-class checkpoints
    m.load_state_dict(ck["state_dict"])
    m.to(_device).eval()
    _model, _classes = m, classes


def classify_tackle(image: str | Path | bytes) -> tuple[str, float]:
    """-> (class name, probability of that label)."""
    if _model is None:
        load_classifier()
    img = Image.open(io.BytesIO(image) if isinstance(image, (bytes, bytearray))
                     else image).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(_model(_tf(img)[None].to(_device))[0], dim=0)
    i = int(probs.argmax())
    return _classes[i], float(probs[i])


def classifier_classes() -> list[str]:
    if _model is None:
        load_classifier()
    return list(_classes)


def predict_proba_bgr(bgr_list, batch_size: int = 32):
    """Batched local inference on a list of BGR numpy frames (no API cost).

    Returns (probs, classes): probs is an (N, C) array of softmax
    probabilities aligned to the returned class-name list. Used by the video
    judge to score EVERY frame of a clip locally.
    """
    import cv2
    import numpy as np

    if _model is None:
        load_classifier()
    out = []
    for i in range(0, len(bgr_list), batch_size):
        chunk = bgr_list[i:i + batch_size]
        xs = torch.stack([
            _tf(Image.fromarray(cv2.cvtColor(b, cv2.COLOR_BGR2RGB))) for b in chunk
        ]).to(_device)
        with torch.no_grad():
            out.append(torch.softmax(_model(xs), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.empty((0, len(_classes))), list(_classes)


In [ ]:
%%writefile src/airef/handball_classifier.py
"""Handball spotter — third opinion in the foul-checker judge.

ResNet-18 fine-tuned (locally, 2026-07-18) on the team-provided
Football-Soccer-handball-classifier dataset (109 handball / 107 clean train
images) plus tackle and clean-challenge frames as extra no-handball negatives
(the 4x-larger negative pool is what cut false handball calls on tackle
images from 28% to ~5%).

Verified: 92.3% three-way balanced accuracy (handball 91% / clean 90% /
tackles-as-clean 95%), on small test sets (46+42+100) — expect ~±5% noise.

HONEST LIMIT: the training data has no goalkeeper labels, so this model
cannot apply the goalkeeper exemption, and no single frame shows intent or
deliberate-touch context. Treat every handball flag as needs-human-review.
"""
from __future__ import annotations

import io
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

from .device import pick_device

DEFAULT_WEIGHTS = Path(__file__).resolve().parents[2] / "runs/handball/resnet18_handball_mixed.pt"

_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_model: nn.Module | None = None
_classes: list[str] = []
_device = "cpu"


def load_handball(weights: str | Path = DEFAULT_WEIGHTS) -> None:
    global _model, _classes, _device
    weights = Path(weights)
    if not weights.exists():
        raise FileNotFoundError(
            f"handball weights not found at {weights} — "
            "copy runs/handball/resnet18_handball_mixed.pt there first."
        )
    _device = pick_device()
    ck = torch.load(weights, map_location=_device, weights_only=False)
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 2)
    m.load_state_dict(ck["state_dict"])
    m.to(_device).eval()
    _model, _classes = m, list(ck["classes"])


def classify_handball(image: str | Path | bytes) -> tuple[str, float]:
    """-> ('handball' | 'no_handball', probability of that label)."""
    if _model is None:
        load_handball()
    img = Image.open(io.BytesIO(image) if isinstance(image, (bytes, bytearray))
                     else image).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(_model(_tf(img)[None].to(_device))[0], dim=0)
    i = int(probs.argmax())
    return _classes[i], float(probs[i])


In [ ]:
%%writefile src/airef/contact.py
"""Contact gate + ball evidence — the missing 'is anything even happening?' check.

Approximates the user-level rule (contact on the BALL first = clean; contact
on the BODY = foul) at the level our models can actually see:

- CONTACT GATE: a foul/clean verdict is only meaningful if a fouling_player
  box and a victim box actually touch or overlap. Players standing apart =
  NO INCIDENT, whatever the classifiers hallucinate.
- BALL EVIDENCE: an off-the-shelf YOLOv8n (COCO 'sports ball', runs locally,
  no API cost) locates the ball. Ball inside the challenge zone supports a
  clean read; ball far from the contact supports a foul read. Also gates the
  handball flag: no ball near any player -> handball score is suppressed.

Honest limit: this is box geometry, not literal first-contact. True
'cleat-touched-ball-before-shin' needs pose keypoints or contact-labeled
training data — the next rung on the ladder, not this one.
"""
from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np

_yolo = None


def _ball_model():
    global _yolo
    if _yolo is None:
        from ultralytics import YOLO
        _yolo = YOLO("yolov8n.pt")   # auto-downloads (~6 MB) on first use
    return _yolo


def _xyxy(d):
    return (d.x - d.width / 2, d.y - d.height / 2,
            d.x + d.width / 2, d.y + d.height / 2)


def _gap(a, b) -> float:
    """Negative = boxes overlap (by that many px); positive = separation px."""
    ox = min(a[2], b[2]) - max(a[0], b[0])
    oy = min(a[3], b[3]) - max(a[1], b[1])
    if ox > 0 and oy > 0:
        return -float(min(ox, oy))
    dx = max(max(b[0] - a[2], a[0] - b[2]), 0.0)
    dy = max(max(b[1] - a[3], a[1] - b[3]), 0.0)
    return float(np.hypot(dx, dy))


def _overlap_ratio(a, b) -> float:
    """Intersection area / smaller box area. Side-by-side players overlap a
    little; tangled players overlap a lot. This separates them where a raw
    'do the boxes touch' test cannot."""
    ox = max(min(a[2], b[2]) - max(a[0], b[0]), 0.0)
    oy = max(min(a[3], b[3]) - max(a[1], b[1]), 0.0)
    inter = ox * oy
    if inter == 0.0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / max(min(area_a, area_b), 1.0)


@dataclass
class ContactInfo:
    in_contact: bool
    gap_body: float | None        # closest fouling_player↔victim gap in body-heights (negative = overlap)
    overlap: float = 0.0          # intersection / smaller-box area of the closest pair
    ball_found: bool = False
    ball_xy: tuple | None = None
    ball_in_challenge: bool = False
    ball_dist_body: float | None = None  # ball distance from the contact zone centre


def analyze_contact(img_bgr, detections, min_overlap: float = 0.15,
                    use_roboflow_ball: bool = True) -> ContactInfo:
    fps = [d for d in detections if d.cls == "fouling_player"]
    vics = [d for d in detections if d.cls == "victim"]
    body = float(np.median([d.height for d in fps + vics])) if fps + vics else 1.0

    gap_body, pair, overlap = None, None, 0.0
    for f in fps:
        for vv in vics:
            g = _gap(_xyxy(f), _xyxy(vv)) / max(body, 1.0)
            if gap_body is None or g < gap_body:
                gap_body, pair = g, (f, vv)
                overlap = _overlap_ratio(_xyxy(f), _xyxy(vv))
    # REAL contact needs substantial box overlap. Two players standing
    # side by side have tall boxes that overlap a little (~5-10%) without
    # touching; genuinely tangled players overlap far more. A raw "boxes
    # touch" test (the old rule) mislabels the side-by-side case as contact.
    in_contact = overlap >= min_overlap

    def _detect_ball(im, conf_thr):
        try:
            res = _ball_model()(im, verbose=False, conf=conf_thr)[0]
            balls = [b for b in res.boxes if int(b.cls) == 32]  # COCO sports ball
            if not balls:
                return None
            bb = max(balls, key=lambda b: float(b.conf))
            x0, y0, x1, y1 = map(float, bb.xyxy[0])
            return float(bb.conf), ((x0 + x1) / 2, (y0 + y1) / 2)
        except Exception:
            return None                                 # ball evidence is optional

    # PRIMARY: the team's soccer-specific RF-DETR ball model (hosted). It sees
    # balls the generic COCO detector cannot — measured 2026-07-20, a ball
    # pressed against an arm scores 0.51 here vs 0.13 (noise) on COCO.
    # One API call; falls back to the local COCO detector if it's unavailable.
    found = None
    if use_roboflow_ball:
        # Soccer-specific detector ONLY — no COCO fallback. Measured
        # 2026-07-20: when this model finds nothing, COCO "recovers" the miss
        # by labelling white shorts as a ball. A false ball is worse than no
        # ball (it corrupts ball-in-challenge and the handball gate), so we
        # report honestly instead of guessing.
        from .roboflow_client import RoboflowError, detect_ball
        ok, buf = cv2.imencode(".jpg", img_bgr)
        try:
            b = detect_ball(buf.tobytes(), threshold=0.40)
            if b is not None:
                found = (b.confidence, (b.x, b.y))
        except RoboflowError:
            found = None      # offline/API failure -> simply "no ball evidence"
            # NOTE: deliberately narrow. A broad `except` here once swallowed a
            # NameError and silently degraded every call to the COCO fallback.
    else:
        # Offline path (no API): local COCO at >= 0.20. Below that it invents
        # balls in ball-free scenes, so low-confidence hits are ignored.
        found = _detect_ball(img_bgr, 0.20)
    if found is None:
        h, w = img_bgr.shape[:2]
        for d in (fps + vics)[:4]:
            x0, y0, x1, y1 = _xyxy(d)
            m = 0.5 * d.height
            cx0, cy0 = int(max(0, x0 - m)), int(max(0, y0 - m))
            cx1, cy1 = int(min(w, x1 + m)), int(min(h, y1 + m))
            if cx1 - cx0 < 40 or cy1 - cy0 < 40:
                continue
            sub = _detect_ball(img_bgr[cy0:cy1, cx0:cx1], 0.20)
            if sub is not None and (found is None or sub[0] > found[0]):
                found = sub[0], (sub[1][0] + cx0, sub[1][1] + cy0)
    ball_found = found is not None
    ball_xy = found[1] if ball_found else None

    ball_in, ball_dist = False, None
    if ball_found and pair is not None:
        f, vv = pair
        fa, va = _xyxy(f), _xyxy(vv)
        zx0, zy0 = min(fa[0], va[0]), min(fa[1], va[1])
        zx1, zy1 = max(fa[2], va[2]), max(fa[3], va[3])
        mx, my = 0.10 * (zx1 - zx0), 0.10 * (zy1 - zy0)
        ball_in = (zx0 - mx) <= ball_xy[0] <= (zx1 + mx) and \
                  (zy0 - my) <= ball_xy[1] <= (zy1 + my)
        cx, cy = (zx0 + zx1) / 2, (zy0 + zy1) / 2
        ball_dist = float(np.hypot(ball_xy[0] - cx, ball_xy[1] - cy)) / max(body, 1.0)
    return ContactInfo(in_contact, gap_body, overlap, ball_found, ball_xy,
                       ball_in, ball_dist)


def ball_near_players(info: ContactInfo, detections) -> bool:
    """Handball gating: is the ball near ANY detected player box?"""
    if not info.ball_found:
        return False
    bx, by = info.ball_xy
    for d in detections:
        x0, y0, x1, y1 = _xyxy(d)
        m = 0.4 * d.height
        if (x0 - m) <= bx <= (x1 + m) and (y0 - m) <= by <= (y1 + m):
            return True
    return False


In [ ]:
%%writefile src/airef/contact_classifier.py
"""Contact-type classifier: player_to_ball / player_to_player / no_contact.

ResNet-34 fine-tuned (locally, 2026-07-19). Trained on a proxy mapping of
existing data — clean tackles + handballs = ball contact; fouls = player
contact; broadcast idle frames = no contact — with a per-source held-out check
confirming it learned the concept, not each source's camera style.

Verified held-out precision: ball 92.9% / player 92.6% / no_contact 98.8%.

HONEST LIMIT: labels are proxies (no one hand-marked literal contact type), and
this reads a whole frame, not the pixel where a boot meets ball vs shin. It is
a strong prior on WHAT KIND of contact a frame shows, not a physics-level
first-contact detector.
"""
from __future__ import annotations

import io
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

from .device import pick_device

DEFAULT_WEIGHTS = Path(__file__).resolve().parents[2] / "runs/contact/resnet_contact.pt"

_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

_model: nn.Module | None = None
_classes: list[str] = []
_device = "cpu"


def _build(arch: str, n: int) -> nn.Module:
    if arch == "resnet34":
        m = models.resnet34(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n))
    else:
        m = models.resnet18(weights=None)
        m.fc = nn.Linear(m.fc.in_features, n)
    return m


def load_contact(weights: str | Path = DEFAULT_WEIGHTS) -> None:
    global _model, _classes, _device
    weights = Path(weights)
    if not weights.exists():
        raise FileNotFoundError(
            f"contact weights not found at {weights} — train them first "
            "(see the training cell) or copy resnet_contact.pt there."
        )
    _device = pick_device()
    ck = torch.load(weights, map_location=_device, weights_only=False)
    _classes = list(ck["classes"])
    m = _build(ck.get("arch", "resnet34"), len(_classes))
    m.load_state_dict(ck["state_dict"])
    m.to(_device).eval()
    _model = m


def classify_contact(image: str | Path | bytes) -> tuple[str, float]:
    """-> (contact type, probability of that label)."""
    if _model is None:
        load_contact()
    img = Image.open(io.BytesIO(image) if isinstance(image, (bytes, bytearray))
                     else image).convert("RGB")
    with torch.no_grad():
        probs = torch.softmax(_model(_tf(img)[None].to(_device))[0], dim=0)
    i = int(probs.argmax())
    return _classes[i], float(probs[i])


def contact_classes() -> list[str]:
    if _model is None:
        load_contact()
    return list(_classes)


def predict_proba_bgr(bgr_list, batch_size: int = 32):
    """Batched local inference on BGR frames -> (probs (N,C), classes)."""
    import cv2
    import numpy as np

    if _model is None:
        load_contact()
    out = []
    for i in range(0, len(bgr_list), batch_size):
        chunk = bgr_list[i:i + batch_size]
        xs = torch.stack([
            _tf(Image.fromarray(cv2.cvtColor(b, cv2.COLOR_BGR2RGB))) for b in chunk
        ]).to(_device)
        with torch.no_grad():
            out.append(torch.softmax(_model(xs), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.empty((0, len(_classes))), list(_classes)


In [ ]:
%%writefile src/airef/video_judge.py
"""Video judge: analyze EVERY frame, itemize what happened, flag contact.

Design (the "smart" part — cost without waste):
  PASS 1  every frame, LOCAL only, 0 API calls:
          contact-type classifier (player_to_ball / player_to_player / no_contact)
          + tackle classifier (foul / clean / no_incident).
  PASS 2  segment the clip into runs of the same reading, then spend ONE API
          call on the PEAK frame of each segment that flags contact — the same
          full image analysis the photo checker runs (foul detector + ball).
          Quiet segments cost nothing. Capped by max_events.

Output is an itemized report: a timeline of every segment, then a detailed
verdict for each contact event, then a one-line summary.
"""
from __future__ import annotations

import io
from dataclasses import dataclass, field

import cv2
import numpy as np

from .contact import analyze_contact, ball_near_players
from .contact_classifier import contact_classes
from .contact_classifier import predict_proba_bgr as contact_proba_bgr
from .handball_classifier import classify_handball
from .roboflow_client import FOUL_CLASSES, run_foul_workflow
from .tackle_classifier import classifier_classes, classify_tackle
from .tackle_classifier import predict_proba_bgr as tackle_proba_bgr

MAX_SEND_W = 640
CONTACT_CLASSES = ("player_to_player_contact", "player_to_ball_contact")


def _all_frames(video, max_frames: int):
    """Decode every frame -> [(t, bgr)], downscaled; evenly sampled if too many."""
    import av

    src = io.BytesIO(bytes(video)) if isinstance(video, (bytes, bytearray)) else str(video)
    out = []
    with av.open(src) as c:
        tb = c.streams.video[0].time_base
        for f in c.decode(video=0):
            if f.pts is None:
                continue
            img = f.to_ndarray(format="bgr24")
            h, w = img.shape[:2]
            if w > MAX_SEND_W:
                img = cv2.resize(img, (MAX_SEND_W, int(h * MAX_SEND_W / w)))
            out.append((float(f.pts * tb), img))
    if len(out) > max_frames:
        idx = np.linspace(0, len(out) - 1, max_frames).round().astype(int)
        out = [out[i] for i in idx]
    return out


def _incident_crop(img, dets):
    boxes = [d for d in dets if d.cls in FOUL_CLASSES or d.cls == "victim"]
    if not boxes:
        return img
    h, w = img.shape[:2]
    x0 = min(d.x - d.width / 2 for d in boxes); y0 = min(d.y - d.height / 2 for d in boxes)
    x1 = max(d.x + d.width / 2 for d in boxes); y1 = max(d.y + d.height / 2 for d in boxes)
    mx, my = 0.2 * (x1 - x0), 0.2 * (y1 - y0)
    x0, y0 = int(max(0, x0 - mx)), int(max(0, y0 - my))
    x1, y1 = int(min(w, x1 + mx)), int(min(h, y1 + my))
    return img[y0:y1, x0:x1] if (x1 - x0 >= 50 and y1 - y0 >= 50) else img


def _handball_prob(img, dets) -> float:
    h, w = img.shape[:2]
    views = [img]
    for d in dets:
        bw, bh = d.width * 1.8, d.height * 1.4
        x0, y0 = int(max(0, d.x - bw / 2)), int(max(0, d.y - bh / 2))
        x1, y1 = int(min(w, d.x + bw / 2)), int(min(h, d.y + bh / 2))
        if x1 - x0 >= 40 and y1 - y0 >= 40:
            views.append(img[y0:y1, x0:x1])
    best = 0.0
    for v in views:
        ok, buf = cv2.imencode(".jpg", v)
        cls, p = classify_handball(buf.tobytes())
        best = max(best, p if cls == "handball" else 1 - p)
    return best


@dataclass
class Segment:
    t_start: float
    t_end: float
    reading: str          # dominant contact-type in this stretch
    conf: float
    flagged: bool         # did it flag contact (worth an API call)?
    peak_t: float = 0.0
    verdict: str | None = None
    detail: str | None = None


@dataclass
class VideoVerdict:
    timeline: list = field(default_factory=list)     # [(t, P(contact))]
    segments: list = field(default_factory=list)     # [Segment]
    peak_t: float = 0.0
    peak_frame: np.ndarray | None = None
    peak_detections: list = field(default_factory=list)
    report: str = ""
    n_calls: int = 0
    is_foul: bool = False


def judge_video(video, max_frames=400, max_events=6, contact_thr=0.55,
                min_seg_seconds=0.15, progress=print) -> VideoVerdict:
    """Every frame analyzed locally; API spent only on flagged contact moments."""
    v = VideoVerdict()

    progress("decoding every frame...")
    frames = _all_frames(video, max_frames)
    if not frames:
        raise ValueError("no decodable frames in the video")
    times = np.array([t for t, _ in frames])
    imgs = [f for _, f in frames]
    dt = float(np.median(np.diff(times))) if len(times) > 1 else 0.04

    progress(f"analyzing all {len(frames)} frames locally (0 API calls)...")
    cprobs, cclasses = contact_proba_bgr(imgs)
    cci = {c: i for i, c in enumerate(cclasses)}
    tprobs, tclasses = tackle_proba_bgr(imgs)
    tci = {c: i for i, c in enumerate(tclasses)}

    ctop = cprobs.argmax(1)
    contact_score = np.zeros(len(frames))
    for name in CONTACT_CLASSES:
        if name in cci:
            contact_score = np.maximum(contact_score, cprobs[:, cci[name]])
    v.timeline = [(float(times[i]), float(contact_score[i])) for i in range(len(frames))]

    # ---- segment the clip into runs of the same dominant reading ----
    labels = [cclasses[i] for i in ctop]
    segs_raw, start = [], 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[start]:
            segs_raw.append((start, i - 1))
            start = i
    min_len = max(1, int(round(min_seg_seconds / max(dt, 1e-3))))

    for a, b in segs_raw:
        if (b - a + 1) < min_len:
            continue
        reading = labels[a]
        conf = float(np.mean(cprobs[a:b + 1, cci[reading]]))
        flagged = reading in CONTACT_CLASSES and conf >= contact_thr
        peak_i = a + int(np.argmax(contact_score[a:b + 1])) if flagged else a
        v.segments.append(Segment(float(times[a]), float(times[b]), reading, conf,
                                  flagged, float(times[peak_i])))

    # ---- PASS 2: one API call per flagged segment (strongest first, capped) ----
    flagged = [s for s in v.segments if s.flagged]
    flagged.sort(key=lambda s: -s.conf)
    to_analyze = flagged[:max_events]
    progress(f"{len(flagged)} contact segment(s) flagged; "
             f"analyzing {len(to_analyze)} in detail ({len(to_analyze)} API calls)...")

    fouls = []
    for seg in to_analyze:
        i = int(np.argmin(np.abs(times - seg.peak_t)))
        img = imgs[i]
        ok, buf = cv2.imencode(".jpg", img)
        dets, _ = run_foul_workflow(buf.tobytes())
        v.n_calls += 1
        info = analyze_contact(img, dets)
        ok, cbuf = cv2.imencode(".jpg", _incident_crop(img, dets))
        tcls, tp = classify_tackle(cbuf.tobytes())
        hb_p = _handball_prob(img, dets)
        det_str = ", ".join(f"{d.cls} {d.confidence:.2f}"
                            for d in sorted(dets, key=lambda d: -d.confidence)[:3]) or "none"
        ball_str = ("ball in challenge" if info.ball_in_challenge
                    else "ball found, away from contact" if info.ball_found
                    else "no ball detected")
        geo = "boxes overlap" if info.in_contact else (
            f"{info.gap_body:.1f} body-heights apart" if info.gap_body is not None
            else "no player pair")

        if seg.reading == "player_to_ball_contact" and hb_p >= 0.75:
            seg.verdict = f"HANDBALL ({max(hb_p, seg.conf):.0%})"
        elif not info.in_contact:
            seg.verdict = "NO FOUL — players not actually touching"
        elif seg.reading == "player_to_ball_contact":
            seg.verdict = f"NO FOUL — ball contact / ball won ({seg.conf:.0%})"
        elif tcls == "foul":
            conf = (seg.conf + tp) / 2
            seg.verdict = f"FOUL ({conf:.0%}) — player-to-player contact"
            fouls.append((seg.peak_t, conf))
        else:
            seg.verdict = f"NO FOUL — contact but reads {tcls} ({tp:.0%})"
        seg.detail = f"detector: {det_str} | {ball_str} | {geo}"

        if fouls and seg.peak_t == fouls[-1][0]:
            v.peak_t, v.peak_frame, v.peak_detections = seg.peak_t, img, dets
            v.is_foul = True
        elif v.peak_frame is None:
            v.peak_t, v.peak_frame, v.peak_detections = seg.peak_t, img, dets

    if v.peak_frame is None:                      # nothing flagged at all
        i = int(np.argmax(contact_score))
        v.peak_t, v.peak_frame = float(times[i]), imgs[i]

    # ---- itemized report ----
    L = [f"FRAME-BY-FRAME SWEEP — {len(frames)} frames analyzed locally (0 API calls)",
         "",
         "TIMELINE"]
    if not v.segments:
        L.append("  (clip too short to segment)")
    for s in sorted(v.segments, key=lambda s: s.t_start):
        mark = "  ** CONTACT FLAGGED" if s.flagged else ""
        nice = s.reading.replace("_", " ")
        L.append(f"  {s.t_start:5.1f}-{s.t_end:5.1f}s   {nice:26s} {s.conf:3.0%}{mark}")

    analyzed = [s for s in v.segments if s.verdict]
    L += ["", f"CONTACT EVENTS ANALYZED IN DETAIL — {len(analyzed)} "
              f"({v.n_calls} API call{'s' if v.n_calls != 1 else ''})"]
    if not analyzed:
        L.append("  none — no segment flagged contact above threshold")
    for n, s in enumerate(sorted(analyzed, key=lambda s: s.t_start), 1):
        L.append(f"  [{n}] t={s.peak_t:.1f}s   {s.verdict}")
        L.append(f"        {s.detail}")

    L += ["", "SUMMARY"]
    if fouls:
        best_t, best_c = max(fouls, key=lambda f: f[1])
        L.append(f"  {len(fouls)} foul(s) found — strongest at t={best_t:.1f}s ({best_c:.0%})")
    elif analyzed:
        L.append("  contact occurred but NO FOUL called anywhere in the clip")
    else:
        L.append("  NO CONTACT anywhere in the clip — nothing to call")
    v.report = "\n".join(L)
    return v


## 2b. Load classifier weights (3 files)
Put `resnet18_mixed.pt` (tackle), `resnet18_handball_mixed.pt` (handball),
and `resnet_contact.pt` (contact-type) in your Drive at
`MyDrive/ai_referee_runs/`. Falls back to a manual upload prompt per file.

In [ ]:
import shutil
from pathlib import Path

WEIGHTS = {'runs/clean_tackle/resnet18_mixed.pt': 'resnet18_mixed.pt',
           'runs/handball/resnet18_handball_mixed.pt': 'resnet18_handball_mixed.pt',
           'runs/contact/resnet_contact.pt': 'resnet_contact.pt'}
drive_dir = Path('/content/drive/MyDrive/ai_referee_runs')
missing = {k: v for k, v in WEIGHTS.items() if not Path(k).exists()}
if missing:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive not available:', e)
for local, name in WEIGHTS.items():
    local = Path(local)
    local.parent.mkdir(parents=True, exist_ok=True)
    if not local.exists() and (drive_dir / name).exists():
        shutil.copy(drive_dir / name, local)
        print(f'{name}: copied from Drive')
    if not local.exists():
        print(f'{name} not in Drive — upload it now:')
        from google.colab import files
        for _, data in files.upload().items():
            local.write_bytes(data)

from airef.tackle_classifier import load_classifier
from airef.handball_classifier import load_handball
from airef.contact_classifier import load_contact
load_classifier('runs/clean_tackle/resnet18_mixed.pt')
load_handball('runs/handball/resnet18_handball_mixed.pt')
load_contact('runs/contact/resnet_contact.pt')
print('tackle + handball + contact-type classifiers ready')

## 3. The judge

In [ ]:
import cv2, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from airef.roboflow_client import FOUL_CLASSES, classify_foul
from airef.tackle_classifier import classify_tackle
from airef.handball_classifier import classify_handball
from airef.contact import analyze_contact, ball_near_players
from airef.contact_classifier import classify_contact

THRESHOLD = 0.5    # detector foul-box confidence bar
OVERRULE = 0.75    # classifier confidence needed to overrule the detector
HB_FLAG = 0.75     # handball likelihood that triggers a review flag
REVIEW_BELOW = 0.70  # verdicts under this confidence get a double-check line

def _draw_label(img, text, x0, y0, color, taken):
    """Readable label near (x0, y0): filled background, flips inside the box
    when the top edge is too close, and shifts down to dodge other labels."""
    h, w = img.shape[:2]
    fs = max(0.32, min(0.7, w / 1100.0))      # scale text to image size
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, fs, 1)
    x = int(max(2, min(x0, w - tw - 8)))
    y = int(y0 - 8) if y0 - 8 - th > 2 else int(y0 + th + 10)
    rect = (x, y - th - 4, x + tw + 6, y + 4)
    def hits(r):
        return any(not (r[2] <= t[0] or t[2] <= r[0] or r[3] <= t[1] or t[3] <= r[1])
                   for t in taken)
    while hits(rect) and rect[3] < h - 4:
        y += th + 12
        rect = (x, y - th - 4, x + tw + 6, y + 4)
    taken.append(rect)
    cv2.rectangle(img, rect[:2], rect[2:], color, -1)
    cv2.putText(img, text, (x + 3, y), cv2.FONT_HERSHEY_SIMPLEX, fs,
                (255, 255, 255), 1, cv2.LINE_AA)

def _handball_prob(arr_bgr, detections):
    """Max handball likelihood over the full frame AND crops around each
    detected player — the classifier is strongest on tight views."""
    h, w = arr_bgr.shape[:2]
    views = [arr_bgr]
    for d in detections:
        bw, bh = d.width * 1.8, d.height * 1.4
        x0, y0 = int(max(0, d.x - bw / 2)), int(max(0, d.y - bh / 2))
        x1, y1 = int(min(w, d.x + bw / 2)), int(min(h, d.y + bh / 2))
        if x1 - x0 >= 40 and y1 - y0 >= 40:
            views.append(arr_bgr[y0:y1, x0:x1])
    best = 0.0
    for v in views:
        ok, buf = cv2.imencode('.jpg', v)
        cls, pr = classify_handball(buf.tobytes())
        best = max(best, pr if cls == 'handball' else 1 - pr)
    return best

def judge(image, label='image'):
    """Run one image through the model, draw its boxes, print a verdict."""
    data = bytes(image) if isinstance(image, (bytes, bytearray)) \
        else Path(image).read_bytes()
    arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if arr is None:
        print(f'{label}: not a readable image — skipped')
        return
    call = classify_foul(data, threshold=THRESHOLD)
    # run the contact/ball analysis BEFORE drawing so the ball can be shown
    info = analyze_contact(arr, call.detections)

    img = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    taken = []
    for d in sorted(call.detections, key=lambda d: -d.confidence):
        x0, y0 = int(d.x - d.width / 2), int(d.y - d.height / 2)
        x1, y1 = int(d.x + d.width / 2), int(d.y + d.height / 2)
        color = (255, 60, 60) if d.cls in FOUL_CLASSES else (60, 120, 255)
        cv2.rectangle(img, (x0, y0), (x1, y1), color, 2)
        _draw_label(img, f'{d.cls} {d.confidence:.2f}', x0, y0, color, taken)
    # the ball (soccer-specific detector) — yellow circle
    if info.ball_found and info.ball_xy is not None:
        bx, by = int(info.ball_xy[0]), int(info.ball_xy[1])
        r = max(10, int(0.03 * max(img.shape[:2])))
        cv2.circle(img, (bx, by), r, (255, 210, 0), 3)
        _draw_label(img, 'ball', bx - r, by - r, (255, 210, 0), taken)
    plt.figure(figsize=(8, 8)); plt.imshow(img); plt.axis('off')
    plt.title(label); plt.show()

    print('=' * 56)
    if not call.detections:
        print(f'{label}: nothing recognized. The detector only knows\n'
              f'foul scenes ({sorted(FOUL_CLASSES)} + victim) — on any other\n'
              'kind of picture, finding nothing is the correct answer.')
        print('=' * 56)
        return

    # crop to the incident so the classifiers see what they were trained on
    crop = arr
    fb = [d for d in call.detections if d.cls in FOUL_CLASSES or d.cls == 'victim']
    if fb:
        h, w = arr.shape[:2]
        x0 = int(max(0, min(d.x - d.width/2 for d in fb) - 10))
        y0 = int(max(0, min(d.y - d.height/2 for d in fb) - 10))
        x1 = int(min(w, max(d.x + d.width/2 for d in fb) + 10))
        y1 = int(min(h, max(d.y + d.height/2 for d in fb) + 10))
        if x1-x0 >= 50 and y1-y0 >= 50:
            crop = arr[y0:y1, x0:x1]
    cbytes = cv2.imencode('.jpg', crop)[1].tobytes()
    contact_type, ctp = classify_contact(cbytes)
    tackle_cls, p = classify_tackle(cbytes)
    hb_p = _handball_prob(arr, call.detections)
    hb_valid = hb_p >= HB_FLAG and ball_near_players(info, call.detections)
    geo = ('boxes overlap' if info.in_contact else
           f'{info.gap_body:.1f} body-heights apart' if info.gap_body is not None
           else 'no player pair')
    print(f'{label}:')
    print(f'  contact-type : {contact_type} ({ctp:.0%})   [geometry: {geo}]')
    print(f'  tackle head  : {tackle_cls} ({p:.0%}) | handball: {hb_p:.0%}')
    review = None
    ball_is_hand = contact_type == 'player_to_ball_contact' and hb_p >= HB_FLAG
    # contact-type + handball decide the KIND; geometry gates whether there
    # is contact at all (boxes apart = can't be a foul, whatever else says)
    if ball_is_hand:
        verdict = f'HANDBALL ({max(hb_p, ctp):.0%} confident)'
        review = 'keeper status and intent are not visible in one frame'
    elif not info.in_contact:
        verdict = 'NO CONTACT — players are not touching; no foul'
    elif contact_type == 'player_to_ball_contact':
        verdict = f'NO FOUL — ball contact / ball won ({ctp:.0%} confident)'
        if tackle_cls == 'foul':
            review = 'tackle head disagrees (reads foul) — worth a look'
    elif contact_type == 'player_to_player_contact':
        if tackle_cls == 'foul':
            conf = (ctp + p + call.confidence) / 3
            verdict = f'FOUL ({conf:.0%} confident) — player-to-player contact'
        else:
            verdict = f'LIKELY FOUL — player-to-player contact ({ctp:.0%}), but '\
                      f'tackle head reads {tackle_cls}'
            review = 'contact-type and tackle head disagree'
    else:
        verdict = f'NO FOUL — no real contact ({ctp:.0%})'
    print(f'  VERDICT: {verdict}')
    if review:
        print(f'  double-check: {review}')
    print('=' * 56)

def safe_judge(image, label='image'):
    """Demo-safe: never lets an error crash the presentation."""
    try:
        judge(image, label=label)
    except Exception as e:
        print(f'\u26a0 could not analyze {label}: {type(e).__name__}: {e}')
        print('  checklist: internet on? cell 1 API key entered? cell 2b ran?')

print('judge() ready — contact-type is the primary decider')

## 3b. PRE-FLIGHT CHECK — run this before you present
Confirms all models are loaded and the Roboflow API answers. If every line
says OK, the demo is ready. If not, it tells you exactly what to fix.

In [ ]:
import numpy as np, cv2
from airef.roboflow_client import run_foul_workflow, detect_ball
ok_all = True
def _check(name, fn):
    global ok_all
    try:
        fn(); print(f'  OK   {name}')
    except Exception as e:
        ok_all = False; print(f'  FAIL {name}: {type(e).__name__}: {e}')
blank = cv2.imencode('.jpg', np.full((64,64,3), 128, np.uint8))[1].tobytes()
_check('Roboflow detector (API + key)', lambda: run_foul_workflow(blank))
_check('tackle classifier', lambda: classify_tackle(blank))
_check('handball classifier', lambda: classify_handball(blank))
_check('contact-type classifier', lambda: classify_contact(blank))
_check('soccer ball detector (API)', lambda: detect_ball(blank))
print('\n' + ('READY TO DEMO' if ok_all else 'NOT READY — fix the FAIL lines above'))

## 3c. Built-in demo images (fallback if a live upload misbehaves)
Three bundled examples — a foul, a handball, and a no-foul — so you always
have something reliable to show. Run `demo()` for all three, or
`demo('foul')` / `demo('handball')` / `demo('clean_or_nofoul')` for one.

In [ ]:
import base64
_DEMO_B64 = {'foul': '...', 'handball': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAYEBQYFBAYGBQYHBwYIChAKCgkJChQODwwQFxQYGBcUFhYaHSUfGhsjHBYWICwgIyYnKSopGR8tMC0oMCUoKSj/2wBDAQcHBwoIChMKChMoGhYaKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCj/wAARCAB/AIEDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDB5wdwx7004z96pCvYke9MwOo2/X0rwCAyPxpr/Nj5e/btSkjuc8UxpAOV44ougALk9xTnjDDbvGO/FR+ccAdc05HG0kBif0oumA1UbIXOR608FFBwzYA5qB5HdjgYFNJ96nmtoBL5vBw+falE7Y+6KgJVTkjr3pdwOcGjmAl3sxPB4pjMSDyKZuIJ5Jz2oV/m+6OvelzAOyTjIHpxSO4xjHT1pAQc9ufWmsNv8OaVgHHy3wCENNaCIsflGfanbVHGcGmnI7VPL3FYi+yJ/tfnRUuWopcsQJ97Nxk0hyeOcVAjNn7y/XNSK74yMU+a4x35j8aMZBXOe2aj81umOc08Nx0wBTuA1SwXHp7U4D3J4pxYsOnFInXAxnvmiwEbHae4pASDnHFTtGuOeaaI8jK4BoAiZw3BUgGhIgpBDe/NTEYyCB60hHr1pNAN2nPOPXNIduME9KcQBk89KAwKjjimgWuxia5e3kLrBplpLcTEZcouQg9a5afU9VsrhZJxKjMD8kh5I9RXdzLfqzf2fGrEqS2XIPFZuvWd9rugoJ1hhu0bgZ5Yf0rtw8oqKUkdtPDpwTtqWdH1FNRtFkXGRwwPrV9y2V746VieE9IXTbedWYyM7de3HpW823OB2rmrcqm1HY45KzsRZ+tFS7PcUVldE3RGybWGVB9qaJAvQc1Dq+opp1nJPNtJ6ID3PpXHR+J7onzJJI8H+ELx9K2hQlOPMilFvY7kLucEnrzUirtPXisrQ9TTU4WaMYeM4dR2960ztX77Y71E48rsxWsPOfUUhHfPXpS4Xbwd30pcdOOvSpENUdec04hh35p8duzsFOEzxlzgVdXRdQYgR27yggYZPmU/iKGpLoWqcmrpGcxOBz+NGTkZH410lt4cEe1tWuVtl7IBuY1R8TWsFhYGfSA0rIM77g8fkOapQbOmlhKk/QoRWkkkEk8jJDAn8chwD7D1NYl3rUttIy2oRf8AawMmucl1PV9amIQTTEcYXhFrPvHvbS98m7ADYBODkVuqKTuj6TAUcLhHytqU31saE2tXFvdi4Y8dWByc1oQa9EyieNTt/iA43H0p9xqA1eFIvsdlG+wIPKt1DNj3rLj090O0gDachCODVNxS8zqr5fKvLmTsjqvDsqXNqBLKgmdiVXOcDPTNaE6GOTa4AP5/rWVLpVvbaAmoRuBJ/s8c96NCvbjULRpp2yitsRvXFc7kql5R6Hm5rl2GpUfa0nZo1Mp60UzP+7RWXN5nzPMznPHUJufD7mFSzRMHOB0FeZxzuSFHQdCK9sKqyMGXKt8pX1ro/AfwBTXW/tS/keysJDmOIDLt/gK9LB1bJwZdNKT12OA8B2sJ0SW5RWWbzQjg9wRkEfr+dbz2+4kZz65r6S0f4c+FdGsEtEtEm8w7Q0jbizdO3eqdx8LPDV9Gr2b3FuCPlKyZB564NFXBznJyKqKDfuHz3HEEBya9X+D/AIW0jW7C41K+jW7lhnMIt5GIRcAZOB1POOeOKn8VfCKW3t2udCuDOEGXgkGGOB1U/wBK4jwnND4ZlfULWWZdYZHE8ErkJI5OSCO3NTSw/JK80TCGtj2HxL8PbO+uLVNNQ2Vs4dZtj/dO3K4B7ZHavCdSTUNC1G6sIbuaB4pCkiPyMiujf4s6jrPjLw1pF1ay6ZAt0J5ism7eFUkDOOnXNYvjS/TVPFOo3cbb0eTAb1xxVYq0Umlqb+2nCFkzGlnv5iWbUp1bvtAqB43lH+lzS3J/2zx+VSlT2IzSFZeOQT6CuBzkYOtOS1YgRQAEwqjoAMCuQ8XxA6hEQMkx5Ndiqvz901yvixC13E7EAKuPcVVHWVzvypc1dHpHw0tNJ8PT2jx6ja3OtXyBijLnyY8ZIGOAT3ya0rnwTb6hdXO4Xtmqyh5LyRhtn3ctsGMYHSvDjDJbr5isWB6Oh4Fatt4k1fYkKajcvGAFEbSkqADkAe1dbatZo+jeDrc7nTnZs2fHelSaSs9laXj3XkrmUMAdhJ4UY74xn61T8GmVtOfeMRq3AA79zVpGlutKnln+eWVizEkksfeneGX/ANEljVQoDVztrlaSPKzCvN89Ko72ND5f7rUUu9vaisreR84d98NPDP2z7Nq1/APs8kwhg3ngYzukI7gfdHvzXvtuzOyxQmNLNVwvlnDYAGPp3ryTwvfmHwrpgGfKih5A7nJz+prsPBOvG/U7vl5wB3xivYw0YwirdTqUVyJF7xD4+8O+HtVGl3M8smoj5vstrAZXGeegHetLwd4k0/xTpsl/pqSIkcrW7rKm11ZeoI7dasw6dp8d/PfLaQi8l5eYoC5wPWvOv2dZ/P0LxAc5I1aUn8VFdO7Ia0PT764FuYGY4Rn2H3yK8a+Nfh3S7/dqulTRf2jER9pjhI+Yf3jjuOlek/Ea3nn8H3xtGKXEa+YhXqCK+ctE125sMlwJWJwwPeufFSsuU1oQjJ+87HPIoS4EzEmZBhXI5FShmUDnHau11XTbXW7P7Zp4WK8UEvEB97HpXFMuGx1559jXjVHJvUzrU5U3yscGOcFhSM2CTuOe1R9896Xnv+FS2YkozxnNc14mtRcXkZklVRtwFA5rfG5mwCSa7q3+BWp6rZpfHV7VJ5EDiPaSBnnGfat8PCU5e6eplNSnSq89V2R4vHpssEZMbOknVSDkN7YohNyHKz/K393YAT/Wu61r4T+MtMnY/wBnNdRr0e3YOD77c5rQ8KfC/wAX6gy+bZmxiJ5efan+Jrf2c9mj7OOMwtubmsvUwNOUmyCEYCnH5im6ZAqWp428mvoHQPhPolhpcw1m6ku5nXb5u/YsZ9V9/r+Vcn4q+FV3pNi11pV1He2qbnYSHZIF69+DUzw1SMW7HyGZ4mniK05UtmeaeWPUUVHuX0k/IUVy+ykeR7M9G8JztN4QtIx97L5x6ZrU+HS3dv4gvN+5rViqx8fdbGT/ADrzPS/FN/p+lLp9mYkjyX8wrljntmvRfhJ4mtxoF4t2xk1BLlpMY5ZSB/LGK9PD1oyUUdUV7qZ7cz+WZEbhWTO49BxXD/A3QLjQNC1RLvrdXhuEO0j5WUY610Nt4js9RtFUWt20h4KLHyf/AK1b1gXFsDMhiGOAx5H1ruRnIkuYVuLaSGQZSRShFfJfi3SptD8R3tlKu0xyHaMdVPINfWrzRqvG5/ZRmvLfjl4Ziv8Aw9Lr0aeVe2ajec8umcc/TNc2LpucOaPQy13R4rpusPaSg5KjI5HY1e1vTxeR/wBo2EbNv5ljjUnB/vYHauUMvzc5x2rtPhp4rh8Naq810pkidDGARnbnvXkwmqjs9DoVdThyzOaaKZU3skiqRwSpFM6cHrXW+LPHrajrs7Qxr9mJ2xqUAGB/LvXJyy+fI543MdxxRUgoaXuZVKcYRTTGDIJwa958L+MIbDSbeKFjcXPlqJGZvljx/CB7dz614MOnpzU+m6xJpV0qD50Y8luxq8PNxuluaYRRc7SPqzStbtdW2QyFYrojcgzyfda4D/hY09rql9Z3TCQ27PGXGBjBwOfWvLNc8QtDDamGeX7duEsCx8MGB657D3rAkmuHvpbkyHMrmVlPILHmu6OK5VaTLxEIU5WR7ZD4lUIdSu5GKQIZSxXIRepCjsT61yK+M9Vv9V+13lwDFISFib7qLnhQK4HXPEWstpYs7S3V42b96AfmYDpzWDFr0kI2TxyQH+6wxg0qtT2i0Zph5wT1Peft+n/8+Vl/3xRXiP8Awk7f3n/OiuT2aOv2sOxmaxdPa7ZEHO3ge9bvwwvNUN+ZtNiM1wC2V27lAPUnPYVg+IITLZZQElTketPj1ebw14auNMtRtvr5f3824gpH/cH171WFgpRSPPpxcqd+iO0sPjlqmmahNb3owiyFfNhVSSAcd66QfH7RIXWVNP1G+ucfenmCrn6CvnG4uWJ+aONu3vWdPJ3VAp9jXqq9zNuzsz6lf9peZVPlaBajHAzcH+grI8U/HO98UaBcaS+mWltHdYV5FdmYDIPGfpXzWZn29cVf0e8ZbyFSflLjOfrVTvysm56i45zjPekOQDjNDSqScYHFA+vFfMNWZjYYjufvJginrMRxnB9aTJz2oGOhxn1pPUZOsxBHORXBeIdWu01+cW0xUKQAvbiu2HJGTivPPE8Xk63OQM7vmH413ZfZzdyoOz0O88PXj3mmRz3D77jlS3StRXGa5rwwjRaSpwQrksB7VsAn14rnxDtUdgk23qXy4AJprxxTxFZFVhjlW5qkHJPXil+07GOe/GazjUaJD+x7D/n3h/M0UfaF9qK09rIfMxk15aW8BErr5vMhDH+EdFHuTXDahemed5ZGBkc5J7CvWtV8Lw3+lLasIw6jmbHIrx3WNOlsdTa0tpfNG7Abpn869LCcqVjuqKUKShbQrTXCgEA557Cn2Gn3GpT7IwSncnoK3LLwwFVWvJmaQ4JVen0zXRRRxwRhIo1RRxhaKuNhDSGpxOV9jzHUrWayuWiuE2uOnoRSWTFZVIxxXT+LdJuLic3cZUxBQNpPSud0+3Mt35bcKoLOfYV20qqqU1IcdT06HbNbxM3UqDxwelSj5QMHNeheHvhHrOr+GINUsp7VPMjDx27sckY4G7pmvP7mCS2nlhnG2WNirjIOCOo4+leDWoTg/eJcbDMk56UgGevSoy2BSeYR0FY2ETEfNyccVgeIf7JjvIpNUhnlZ16Rvt4rcD+vJqKDwZeeMtRdLKSGAW0JdnkP3jnAAA+tdODaVXU2oJuXuq4+2a3a2i+xxtFb7RsVjkgfWpcg96W6099HmNhM6u0GIyy9Dj0qIHuOlY1FeTaIqRlGTTRJnApuTntRuANG8VCiyLC7f9kflRS5HqaKfKwP/9k=', 'clean_or_nofoul': '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAYEBQYFBAYGBQYHBwYIChAKCgkJChQODwwQFxQYGBcUFhYaHSUfGhsjHBYWICwgIyYnKSopGR8tMC0oMCUoKSj/2wBDAQcHBwoIChMKChMoGhYaKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCgoKCj/wAARCADgAY4DASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwD3ufTbC63R6VpWl5VCxaS2QrnsOlYVt4YnuL6G7ubTSIIRkPCtum1PUA46muwhsBp00k0E+I2A3K3PSq5uUtDLJJeR+S+dqNF8wJ+lZX11NY3ILHw5o8VokMem2W1oym7ylJwffHNZV/4S0G0TzRp8DmFQz/ukBK5x6Yq1pmr3U1zCHKBEchhjBYHpitLV7WOfzFcvCZlVfNHPQ+lE0lsU0YFvp+lWlnLHHoQcSSgf6iPoeh4HSmeKfA+n315b3UUdlagYQo8Crnvwa1bi2uru3ngS7ie0ZkUurENGF689yasaliQTQxMQjRBWeQnAPQEe9SkLqZdh4c0+SymgurLTJj5zPb5tUGOc4Pqahj07QZCxuLG0M1vISY4rdVHPGMY6CpLaM22oRXflTssY8pT53ysx4LkelPntVj1CWSJbktuCq6ruVc/e/CmkykyS1tNCtLdIrbTra424LFrZCVUnr06VlzeF4pZLxrbQrG3HnEg7FAkB/irp9Es4rASm48suflVkB5XPQiq3iHVrC3lA1G4igslw++Q4BPTaRVpWIb1ON1Pw3bHS3RNFtEu4SJgvlJllJwTwOlV38Hn+1NXdtM05YjaoY5J412hxyAQoHFatx4o0FrhZrTW9NhFvIEjKqSWQ9vwrokuLe/t1msbm3mE/P2jZvVyOjDHHB7Gs7FcxQt9HjiltHbRdMaKSAeb5dsmFfjI6cj3q/qdtpFjc24OkWL+aBtH2dABzzzjr7Vo6XI62R+1FRclixVGLLn19s+lIkzyRIbcojM2994JIaiyQtXqY9toGmXEkrJY2U8YudzJLaxKQCOR0zxWRdaTFd67bww6Jp0NlDcb2LWiqWRehHFdEBFpepzXc00btOwwjHaVz1q8txN9vkSaeLZg7IwcNg+vvRYcZWZwNzCH1REHh+wFk848+OS2XIX+8DjrmreqaFZadf200ej6XHbFHM6mBCVY9O3Q11+nW06b0YtIiNglxywPP6Uuoo0Fz58h3iRPKSMJ3ouW5GBp/h7So0kRNK09fLVUVBCG4bnOcVa1LRtKs7yxtINI06QCGVwXhQHIXjt3q6Lp57SSNIHhn42jG0gj+ZrH1FrtLOKIyu17Ex2SsMnJ/hPtSvqTuzL0uy0nU9a02K50/TYgbMs8flx5difTGSPeuuPhzRBGYrTTdKjJhKxAWqfKccHpXH3XhqazvNOupSZ5lgeKRwwzg8lfpnpXRqYWk026S7+zxwqINjKR5jemaoUomZfeFIreC4jTT7BkWECMi3TLMBz26dKty+GtIsdPtmuNP09ZZpQ7hoEIVmAyAcdBiuitJpLy7x8piibk5HOR2Hp9aqeJNSsoZVtbmJpXVPNVQBhjnGB780pMlPoc7b6NbwS6nFFomn3dswZYpYoUOGH8LDHufypvhrQrO7j1m1vNF02OSORYQVhXJBXcMnHXkce9a+lrHPObeOZQmnttIIILMRkE+1TRxSwXZaTKXK4IC/dnc9/w6VKLaMnTPDVvYW7W13pNhO0Me1Jzbx/vOpP8AD2GBTNG8MaKbGOOHTbWQvMtwzyW6EliMsBx0HFaN3fTjT1fVkS0ky8Zgb59+eMgjtgnpUnh64shCI/MDCwPkK4YnexA3H6cirYXOXsPCkaazqt7Bptq0I1OIRxmBMsihQx5HTk1Z1jwRp93rElw1hpux5FVI3iCjACjjHfgmulutSsb1ZbdJ3t7iNniQsCAWC5P1GCKgeMLbyG0uobm5falv5jblVwMFj7c/pT5hJmadAsIZSunaRpwknk2EyW6sIwpOGI96t6hY6Vb6XcM+mWEdxE2xd1quDjnjjpipbbz0gvb7y4lmZlhCxuTnBxkH3z+lZd+t68dvNas9xY3MoiBB3MFySefbGOayepTZbsotJvbWJjo+nZdmUH7Mm3AHJBx+FZ6tp1xvistE05p2XGx7ZAygdSeMGtazgW/tZIrWfyYxiJIl+V8A/MfxqrNNcLBdx2Fi4l837MpH3sKME59PepY0yhpvg/SJLaGSawsjdzSmSQLbL90diOwpT4X0S/Cw2mmWYvJXEziS0AURA42HA4rXsdVljk82aK3ZWxEu1vmjA4ww+ta90vmT/aLUiTCCIxjjI7kH2FWnYls42y0GyuTbY0jTmimncAJaLwicYOR0461t/wDCO6RdSQf8SeyTdJtO22QDC9+nSqt7q01ol1JpKRfZLMJBEsmd0pLfOAOuRxg+tbfhHUbvULW5lvEMarKUjjYgsoHr71anZClsGtaXo6W5iWy06OV8Kv8AoyE59hWHH4X0K+vnt7zQ7YLZwAJMLdQshbO4g45x+lR+M9UujqpXT43dLOIyTMjbcE8AZ96b4fudQWx0mzluHM97umAlYFoY855z96ojK4cmhDfaN4dsr8WV7pGni0ijRlka0Uk54AB9M9aZ4Y0jQoRDZPY2F1cW5llKtaIrMc8EZHpxU2q30sunSw3Nv5kt1Oec/K0K9Co7fSpnsrea+uLyKN1uL1EgcjhggAHB6Zq0xpGV4h0fRreG2e80zS1E43yiK0CHg7idygjPTrVG106O9j0i4fSNMixcFgyop3Kc/NIAOmMVs3Ec7XM9g8b3Ee5LWMRxFh5Y+8Wb+E8fjWlqDR21ndJBNFYxpF9lhdlLHJYBcrjPJOKieppF2VihHY+G9PS6fU9I04z3ErkGG1DKVQdRkcVK2meFrixkuH0uxtvtCLHzbJnnoRkVbNkxiksplUrp9sHjaU5Jlx94e3tVqOzFyNO8+8sgkW6SVCTuLkcY9AKIq5DdjMufCmjJJd3c2nWBtI4DA0UtugRQB98cdTwM1R0u20TSLmC1l0m1kt47UbpvKVgG6nt24Ga0NasGu7I6Z9pS/e9yVjVyuVTnr0x65rMFjriaPqRtJGj1CVPs6wSqrmMA8Ec9NuBj2zTsFkay6Fp00FpJollYKtxL5z+faoQV7gcVjwaIbvS5JTY2lu5unAhWziJQDjrjoetaum60NPs7uy1NfLl022DNM3IJxwffJrHt9B1a9a2FjNapBJAbiRZJJN25m4yAfTNEdybNanorWqMPnjk/76zVOWzJlP7smP0Iya18mjd615ksVcSlYylsUVWKqVI5XA7054HkKxszlCueT3rSDc96GbNJYq27Hzsyb3TI7uwe2uAXiZCrKONwIIP86+J/EXhO60T4lr4YvtRmtbZrqNEu5HO0QOflc5OOAeeeoNfdhbHNec/FD4X6X4/ltJ7qSW1urfKrNCQCVJ5DAggjPT8fWvQwOYwjLlm9DCqm1c8d1rx3d/D+ZPBsGkrrGm6cXNvNdnc15vUOHyOcBm42n7uB15r1nxr4g8nw3cR3ROkicJAoaQjBaIPwevBOMe1XtO+HllZ6b4fg2Ws97oiCO0v5YQ0qJ2AHTPfJB/CuI+Lfwz8a/ELV7OZ9S0W306yz5ELNIWYkjLuQuMnA4HAFdqx2Gbs5IxUZc10jmrjV55yyjxq8cRJZEVSAh3BiVAPTcDx6HFS+P/Fser6bZXTX4ubu01N2EETqriE2xjWZN3y5V2ZgCfSteT4W+MpdYstRWTwl9ptkdAxWYgh92R0yPvHnPXP4c5N+zl4h1XxGb7V9X0iG2nl3zCzEjMF4ztDDGeMcmrhisI3rI0tLqYHhvwp4wvfE7+JrHwpef2dePNLDNb3MXmAScKw3PsyOegHU+1eq+D9D8TR+CdU0e8tLjTZbiQyQqsykwiSR3ZVZTn5QUHBFeyaZptrpmmWlhYJ5VraxLFEg7KowKl+yxk5rhrZlzv3Y6G8Gopp9TxObwT4pk+0Qf2nfvDNOJ2dpP3wIZuNwYAfePYcgdq9G8F2Wp6ToEFrq1y9xcpHhmbnLcknvzk11At0Bzk5pRCCwyelZvGuSsSopMxdKkv4Wm/tUrcws37r5ASvPQ/pWpdXUYgd4oA0+35dydT6Zqw0asADggHNLiTzE2svk4O9SOSfrQsRHuNlXS715bVWu9sc56oP4fxqvHfTLqkkdyqTWZwYpQOUPcGrcdmkb5ByOeDUphj/55xn8Kn6zHuN2M29vFS9gMVmJWJJeTP3fQg067ubUXqRy25kGQ3mgZUH3q4LaNXbCqVPIB6Cka1jLKwRRjt2NP28e41oZd7DpSJJm1WUk8AZO4k/y6UlpHYS2j7oQjW5JEa5O098E9TWoLSM7sxg5HY4qL7GRMNq4T+7nmqeIVtB7lnT0ghtmeKIRb/mI7/j71BPptlcajb3s2GuYVOw7uMN6imoiOZo1wXQ8jPI/CsnTrZ4/EF4jwSGDblGfsfY0o1k9ybGncWVnfShQ0gKKyZjbAORjn161PZaVbWrRuhkd492C75znqTTVtV4ITBzk89qm+yRbCQG3f7xrX2kbDkLqVjbXsafaVUlCSpPbise30nT54PK+1ltyk4Q7STIB83HsOK0Y7dyWLswHRe9MXT44W3RHa3HQcnHSo9sFvMqnQrS2ETNdP/oyNuLEEtu7t74yKfo1rYFENi8Xlom0gL+ufXmp1tFfczkrI3XjOcdM02O0WIMi5VSM5UdT70e0QkhiyQae0C3EoljmJZSEwMjnJP5VYaxWSNJbGZIoWJkKouVcnnP/AOqmvZpPGBK5YDoCOlLAkkCrHCdkaDCgdMU/bJDtfYojT5Y7yS9N+m902DdHgKepb61YEiW8FuEvoAQp3Nn72TyR/hU0tubglp2LFAduR0z1qqmk2rPD+4UHkg/3TU+1TC2gmpraXFydPd4vOmTD4T5tw5DZHStC100wwqrzM4UDkDBJ9TVe2sUgu5rqMETu2GYnJYVojzWQYcL+FaRmiWZNxoNl5jXTxuXiLSRhXPBI9Ku6dbLaaaogh2SEFivck+vvUFx5v7yAMxDcE9Kl8u4aHYJlK4xz1rN1U9EOxxV3PdW1rJJNBGsU9yAJiM7x3J9qt2esadd+Yn2mRrqRnit1SPBUgc7D2U+9dBLY/Jbo4DLG+UXOVHHeqdt4ZsrS58+GMhtxbG7jn29KIzSNZS0M2bTLuGVoo41cC2SNGIzh+5U+/el8QxGGyhjmETPbq0hw21RJt4JPWulRCI9nbp9KivNPF3EiTsDGAQQvBOatVCU77nBi6vjrejxb5ra6kj+0yxoMJIiryHP48VpRxL9ls7W7cRiS5Z1bJfoxfHP5da349GggnE0Yld/LMZZn7HtUVz4bs7i0ht5oi8cOdi7zxnrn8alVY9QfkVY72O+0+eO3kWSZw8Lv0C+gYf3gCDU880DzXrPp0pe2Kod0u0ncOo4qxp2hw2Fs8VsrDc5kYlskse+fwqF/DtvLJLJM87SSMCxMhzx0A9qtTXQhruZLa3Z6drdsxt3SMRvGx3FmH3ACB+X4VJcaNBd3FhqU0Bhu2nN2SPl3L2Rn9Mc1qT+H7K5uhNLAzSBNgbccBfpV/VNIi1C1ghneQpF91UYrnjHJocl0GnqcLqs76rHrb6fLHM9xKlnaFiNwH8ZA/u8nFafhp72w17U4Naa0kKBVtti7mWLsCa1NM8L6Zp8ySW9mBIhyrMxJB9q1RpsP26W7VQJ5VCufXFY8zRrzXViYX9qR/wAfMP8A30KcLiJ+VljP0avIsNjh4/8Avqgeep+Uj/gL185ztnGq568JFz94fnS7xXkkb3nZpB/wOrcMt+f+Ws3/AH1Q2w9qeobgTigYHFedRS3ycmWb/vqrSalfpgCV8CjlZXtUd4cdKXA71xS61egcyj8qfHrt4M/OrfVacdGHtDsj1o+lch/wkN2BykePpTT4pljHKpmtPaWEpo7McUVxX/CYsP8Alkpp6+MiesC/nVe3K5onZZpM4rkR4wT+KA/gacvi+E/egf8AA1m676DUonV7sUbya5dfFtoQf3Mgbt71LH4ptG6xyA1l7WTHzROiLGgsQKxYvEVm/USL6cVMNasiR+8b/vmhSlfcLo0w2TThWcuq2WM+dj8KkGq2X/PdRW0W+rHdF4U7qeapf2laYB+0J+dSLfWx6Tx/nWyqW0FdDb/T4boE7pIpcYEsTbWFLa28sLHfcvNwB84GeKkW6hbhZYz9CKQ3MQ/5ax/99Ch1QuiznjrSZAHvVcXMZ4EiH8RTi4I68fWpeJaWg7ol38U3dUQdT0YUufesHipj0JC1GajzRml9an3DQkyPQUcVFzThTWJqMCQYoFNHSlzW8akurE2O4pwao80A1arSXUQ4qCckc0Y9KTPFGaft5FCkZFIdvekJNMZqynipRHbzF4zwKM0zd7Um72rD67UFoSDj2oBGKZn2pCaTxswY/cKMjNR5NGTT+v1O4rEwbmngnFRJUldFPFTavcOUDilBHpSUVXtp9wPHxaxdxUqRovCjFZy3pxzUqXq496zscKNBRz1qVBjuazVvh7VbinZot5GxM/ec7R+tHKVcvJu6hm/OnGVxwJHA+tZrapZxMVMrysP+ea4H5n/CmSa0QCIFSIeo5b8z/TFSI1wZwATIUU/xOwUfrTJLuKI8zvcMOyDA/M/0rnWvg7FpJGdj3JyaPtUZHWobC5r3GpyyLtVI419hk/nWe8jMcsxJqv8AaE9aQzKe9TYVyXNJuOaYsgPcZqWNdx4o5Lhdirk9TUqqxGBmp4oV/iIFXIzEg4wfel7NjTKkNs7EFsgVdjtlXnqad5ykdRQJVz1pqA7koQDHSnfhUJlHrSGXjrVcoXJ/rVe8k8i1nuAhkEKGRlXrtHXFRPPgcHmqtxfMtvcBVVpDGyqrcBs9jVwXvamlKS51zbFyBhNbRTxQmKGVQ6KXDdevP9KkDqDzXIeD7nUoLS7t9Sht4hHKCiRNkrkc59M9q3HmJNKtZS0NcTyqp7mxoy3KrwvX61VluC3c/nVYkk8mlrnbucvMP8xuxI/GlMj5++2PrUQpTQCkyX7RKOkjj8aT7XOD/rZP++qjxkU0pngHmi1x8zLB1C7A4uph/wACNLHqd8xAF1N+dQLbs+AenerUSJAhyQAOSxq4U7hzN6F21vr1QB9qlPtmro1a/wAAfaG/KsxGVlDKQQeQR0NSbqvlsVzNGiNXvgP9cSfpTl1m+HWX9Kzs1DJM0fVMr6iqBTZsrrN+TgTZJOB8ordfTI7ZJtVjKLqbwhZZyPvqOintiuJiu0cgKcE9BXV3mqRTeHSiOzSMADwR+v4V2YeMXGXMWpGePEN+D/yyP4U2XxPdxrlkhz6d6x5plRTkc1lTytI+T+VefKRPtGbcvizUB91kH/ABTV8XagR/yyP/AAHFYLDPWmKOeTWb13J52dH/AMJbqH9yH8qevi2+7pD+RrnMCjOKmwOb7nSnxde/88ov1pP+EvvO8MWfxrnM8UlHKhe0fc6dfF11jmKLP41LH4uuCwBt0P0JrlMZIx1q/ZxY5YcmmoJjU2dUnieXAP2YfnTm8UycAWw/E1zhwOlN71utNEP2jOl/4SiXH/Hsv/fVO/4Sk/8APr/49XNU0nmruP2jOZW0lC7nAij/AL0h2j/GpIm09cia7kZh0CJhc+564/4DWRLI8r7pXZ29WOTTQ3pW6RlubN7cXdvErwxQwWzHAlh+cMfTdWZJJI7Zldnb1Y5Na/hOSX+1Y4QqvaSAi5jflDH3Zuy47Gsq58sXMwhJMQchD6rng0SjdANVsc0M5J60gx3pDjtWXI2Abj60u8+tKsE8pUwiIIOXaRiAq/41Ott60/ZMuVKUUm1oQBmJGM1MiSnuanjiRe2akyoHAqVS7mYkSKuNx5qdJdp4NRDGKXirUEguWVuDjrTvPPrVXIzSinyhctrMc9SacLj3qmHI6GkLgdaTjYVy+Lg+tDXJ7k4rMe5CDiqkk7uepAqGgua0t8F6EE1n3c/2lWUgBSMGqmc9TSqcdKm7Wo07O5JpqpYRmKBflONzHG5yABkn1wB+VX1uj1NZhbJpd5xgGpkmxyk27mst3x0FKLxf4hWQHPrSl6nkFc2VukPrSi6SsVXJ4GasRRswySaagO5ppOHOAatxKMZNZ0OExxVgS8datUx3L24Adar6jNJDp8s0E8cLRFXZpFJXbuwc/iQfzqvJOVHBzWXrk095Yf2daXcdrPdOqlnHLIDuYD34raEbM2wslGonLY6DSrl7y2kupJo5w8mEeNdq4xzge5yaub65nQEOnQTWH20zskrSYPVVP3c//WrVFxnvU1FroaYyUZVW4bGlvFKWUqc4rOW45pWuTWdmcpPJFG2TtAPtV/7f5WkJaYYjzC5OfyFYj3IHJNUbi+aQkA4FHO4rQLmjcThz1qDcD3qgJSR1NNMzdqxabFcvsRUZbnrVIymmibmlZhc0QaU49aoifinefzRZsW5b3UoOTVUTD1pRKdwo5WBoQJuYfWtNQB06Vm2jfJkjmrPnDpmtYQGiwSKCRVYSZOacZBjrVWGT5FM71EGPY0bzTsBxMCTTkeTDI/uF4+pPSrSJbQn/AEucbh1ig+dv++vuj8CayJbu4uBtmmkdewLHA/CkRWz3roTEmbNzqm+0+yW0Yt7Ync6qxLSHsWbqfpVRZATiq6RdzU6qFp7hdEy9eeKs28SSSomVBY4+Y4H41T30Bic4XcTwB61cY3dhw31MvVZorLxDMk3mZukUAibEcWMgHHpznFdONsaLGnCqAvXOeOv41fk+E9zf2l9c3NxYteXa4VZMkW2MfrWTOk1tIYblSsicHIxn3roqU+SOp6ePxirUoQS+FWJ93vRx61SMhz1pfMNch5Rcz70b/eqnmUnm460xF4Nml31nNOAHJJwq7m9APUnsKq/a3aFN8sbSkfOsf3UPoD3ocbK5r7GXJz9DXa4Ud6ryXBI4PWswSnPJp4fjrWbZmW9xbrRmqvmnFIJT61NhFrdSZquJKcJKGgsTA0oNQGQULIDRyjSLAagcnApiEtVmJAMVSgFiSCP1HNWgcVCrACl3jPWqUUBN5lLvOag3D1oDD1osPoTls1E6IXV2RS6Z2seqg9cfXijdQSDSsCutRsMaJdSTgAO6hS3cgdP51OGIqHIx70objOaGrjd3uWFfFNlm2g5NVpJQo4PNU5JSx61MlYknmmLHjvTFYd6hyCKTNY8gFnfxSF81Bk+tPjSSV1SJGd2OFVeST6AU1C4tx26msccngVaa3S1LC/fDgZMMPzOPZj0U/nUL3SBR5FtGvvJ87fjnj9KrkKSCISSnEaO59FUmpzaTqf3nlxe0kqg/lnNVZby4lGHlfb/dBwPyqEe9LkQmXykKn5rtD7KhNSRfZ2cBWuH9TgCs1c54rQt1ZEyc5NNQuBqLNEqgLFL+L0faUB/1A/FjVEPkUF81aVhGiLo/wxxj680G5c9ox/wGs4SU8NmiyHcum4lHRl/IUfaJf7w/Kqe73pQ3vTFc5hLYDqaf5eKXJpa6FBAAFOC0zNKGxVONgHlavaHLHa6pDczDckGZQv8AfYDhfxOKz9xq9pIZbg3JUGO2Hmtu6HngfnTitbjTPXYoxZeCnMzBp/szysxPzBm5/rXjcs81xte4leSTABZjk4Fb+q+IdQv9Ima6eP8A0qRQuwYwi9fzNc3vAHoK6MRPnSRc5XQvI60u7ioXmXH0qu02ciuJxsZltpQBVd56rGRjwelM3HPTNTYe5X8U6kkXhqSHzxDE9yn2oA4aRMEqPXGR+oq4rQGKJrXBhZFZdvv/AFrFvF0/UI7uW40wTNbsUErSECTAyUx6Akc1p6fdG7sLad4YrcyRg+VEMKoxxitZfDY7qte9GNJdCwDTgcU0Uuaw5TguO3YpM0hBYHaCTjoOtY+ka1a6hfTwxWrJOB80jtz8px07dapU21cpK5thvwo34pmDT0j3VPKJoAWarEMR6npT4ogBzUwwOlUoAmKvFPVvWmfSjtWnKIlD0E96iFBNTYLku6lDe9Q5oFTyjJw3FJv96izxRu4oaAl3mmvLgdaiZ8Cq0r7j1qWx3JHlLHrTAxxUWaXNTa4XJd+O9M8w+tMNFOwE6vxzXRaSTb+HNUvrbBvI2SJWxkxI33mHuema5cGtHS9Tk0+4MiKskTKUlhb7sqHqDTha4FPfgngL+FCyDvWle6fHNC15pb+fa5JeP/lrB/ssvcejCsf6c0pRdySxvoD81AGp69RQolFu35fJFXfM96qxqAtPDe9UkCJ94o3g1FminYZMCKcH4qvS0uUWhOH96dv96q0ZPrRykmRuFG4eoqqGzSb66ExFvULGF9KW6vRM8HmBdsbbR7Fj/d9RSLtQBUAVR0A7CuWn1SLUNft7Q30cdoj7djOQuR0J65Oa6IkjOSDz+daVY2PSzHDwpUqajvYs7uM5pTO7Q+SZG8rdv2ZON2MZx61UMgA61WkuDng1i3Y80057tmtoIHcFLfd5Yx03HJ5qm8rN/Ef8aotJk55zS+bxU84XLe8460m6qnmGlMhxSuFyzmiqoc+tPEppXGSCNACoUAE5I9z1qRMKABwo4A9qr+ZxQJe1XcbbZZ3DtS7x61XElG8GoYi7aspuog4LIXUFQcEjNcV4Osnj8U6tuBRopJRjdnB3nqe/FdBqF4bG3Eyr5jlwkcYPLueg/rTtOjtNN1Z7c2jfa7lWla5D5UnuoH6100vhZrHY11Q7uRVhAFHvUYcetMuLmO3haWUkIuMkDoCcf1FQoamcIucuVEwnXf5YZS46rnkVMGzVH7PaQanI0Fq630gZppX3A4UgEAEYGCR0qYSD1qpR5TbEUJUGlItBqC3FV1k96d5nFTYwuSZpC4XqcfWojKKz9csItV0yaOW/lsoYwHdo0DPJyBsUZGWOelVGFy4Rc5KKI9avNXt9QtF0yza7tn4lEYyy+/0rcJIOCDxxXP3bappv9lXsMH2byEWOOKaUNczIo/1rjGQDjB+nFbIujeJ9p3rmU7yq8Yz2q5wsj1MXl6pUI1ou99H5MnzjNMd9oqMvtALGq8su48dK5ZHkDpJSx9qjyaZmjNQwJAwpd1Qk0ZpWC5MH5oLVFmnFs07DuOBp26o80m7mi1hFiG4kt5UlhdkkXoynBH41be/gu/8Aj/twJDx59uAjfiv3T+lZuc0o9DQmBpfYFkG6yuoZx/cY+XJ/3yeD+BNMNpcRYM8EsYPQspAP0qgFy1adpcXFsv7meWPPUK2AfwrRJFXHbuMU0NVr+0A3/Hxb203uY9h/NcU0vZsM+VPGx/uuGA/AjP61VhEQel3VV8QX9po9gLsNdyxBsPiDOz0J9quwWzXNvHPbSwSpIoYbX5OeelNw0uFhoal3CiS2uI/vwyLj1U1CcipsCJy1JuHrUIPrRx60uURgLIfeq9/ex2cHmSK77jsVUGWYnsKmZwpqtKx3Ehh1yMjp9K2SSdwT1OYfw/pFv4kssvcRzkmcRSEEtj1PbFda1xlqzVgQX/2vYfOK7NxYnA9h2qYZ/WlUnzGlSXMTPKSaaDUXNOAPrWDuYtjs80tR4NKCc8iiwiWlxxUY606iwxRS03J9KXOKLWKH9qb3ozScmgBwyelSxx5pI071ZXAHStEgI5bVJUTcoLIwdT3BAxke9JPbec0bOSWjJKnuM9anzinBq0WgXYyMFFwWJxSTufs8qkBlZdrKehXvTiQTTJQHjZD0IxTi9Soy5JKSMnQPFLyeIY3lntZN6vHJCo4kVgoI/JVwa6CYIrZiZthzgMOR+I4I9xXIR+FLXQdU0RJlY38qNI3IVFyu4A+uAa6fcOMGtKjTR2Y/FRxLTSJg2KXdk81FuFIXrBXR56JC7A5QjPvXQ+C9a0vQryS81GITXAG23XdjDE8ke9cwTmnyw2VzpslvcicM5XeEK7ZUDAlDnlcgD5hWtOVmaQdmH7RsOp6t4t0+bTYCYfsKhSSFK/O3JBqjo0J0/TLe3aUyyKg3sepbvWpfWnhiGxhk0OaaO8gkdJt0rMZ0fJBwScFTxgevFZTOD7Zq69S6sazqSa5b6FxpyR14pvm1VLe9ODD1riZgThzTt9Vg1O3cVIMnL0B6r7ieBUFpdpdeYIs7o22MCMEGq5XuNRvqX99OD+9VwTTsmpsIsbqM81ACaUse9FhE+RShuar7qchO6qSGXoFzzVnOOKrQ/KgqQPmrQEtAPPFR7qVWpoDuPAFtpsyXc99FFLNEuEjlI2kHqcdKreLv7JlAn0yMW0yyGN4QMcAcMPY1meFdVtdLvZrjUIzNaiL5o1GTyyjPv16Vv/E5oJDpl1bRtHHNDwrJtwB04rtsvZXNFscalzNH9yaRfoxFTjU7rADzFwP+egDfzrNLZo3Vw3MzSOoAj95a2r+/l7T+mKZ9ptv+fNP+/j/41Q3e9Lupgc+2W5zUbDnk5qQmmmle5FxFUU8AU1RTqB3AKKABmlFHSiyELsHekKgUoNL1o0GMwKMU7FJigQmOKQrTwCTUscWeTzQo3Ajji3VYS3HepUUAU/pWigirkYix0pQuKcTSA1XKAmOaXHY0uaM0rBcYVIph6VIxpp6UCuMkEU0gmuIzLcINsUjMf3Y6Hjuccc9KBQRRQAtFKCaWgEITUcjYGKeTioJG61LKKroA2V470fNjmpOM9KMVDuwuMBp4NN204KKmwXFBp9M2CnA0JAwkbYhYsqgAnLHA4HrW7IBc+DtO1KGCHynwFAJD/e2uWbp96uW1dV+xtM7BEi+ZueorEtfFlvsis5LyQ2jHBh/gY/SuylFOLufQ5XCnKi+a3zOsf5XIyDg4yDkH6GkJqNYFtoooY5zOEQZkxjNLnFcktGeFVSU3y7EitilJzUYNOJ4xSZmKpq1AoJBNVIxlquxkBapICckYpAxApu4Y703NVuBIGJ9qeCMdahoNUI0tEjS41e1hmnkgikbYXj6gnp+tbfiBZpfD9tJLfzTvazG1milbdhuzA+4rlbe4a1uYpljWVkYMEYkA4Oa07nXHnGpoIgsV9N57IwDBWGcY9OprXn92xaloZ2cHmlyKgzzRuINYEk5NBJqIPS76LEmMetIxquJjR52azuiSdW4pymqweniQCk2BYBpCai80EUCQU0xko6UZqMOO9AcetO4iUU5UJojG7FWl2gdKpIoSKLHWpgAO1N3CjdxWiViR9FR7uaC1MZISKbxUZakzSuA8mikBozRcaAUtID2p3BpE3G7aMU4gYphFUO4YpCcUE4qCSQdKTKCV+wqInNITkZoyKhu4xc0goJoB4qRAQT0pRRuxTd1FxJjs0uaZuGaGIx1pNlFXVmU2FwJAShQg4PI96x7fR9Pk0q0SO2eVoWMkW6T7xJz82PwFbjiOQMlwkjwOpVxGcNgjsaZDHFDFHDbrIIkQKvmNk8Dvjj1P41rGfKrFqbirIlsLpbyxhuFAUOvKj+E9x+FWKjjEaIqxQxQqBjbGMDPc/U07dWctSL3HjNO+opqmnDLHg0kgJYVGeasgY6VCgIAqUHjrVpAPHSkNJmkJqloIdSZphalzmhgDdKRTS4FJnnpSQD6KYKcPrTaEwpwNNJHek3CpsJHPDtSioyaXdWAiQ0gpu45ozTAkBo3EU0NTo1aQ8dKEmA5SWNWY4uhIoij2DpzU4bitFEY4AKOKcG461HupCa0sFiYNRuqHdRnFAE26ml+ajyaCc0MQ7fzS78VHQetICQSU5WqGlBwKYyXdShqhBoLUCLG6ms4AqAtUM0uAeaHKw0yaWYYxmqzSZ+lQFyTk0ZrJzbHqTh+OtJvzUINLg0rsbJw9LuxVcEijfSuJEpzSg8VFvNJuNA7E1ITimB6N4qhDgaUUzd7U4Glcdh+aVTzUe4UBuaLoTLCmpIeuarpyQKtouFFXFDRLnigE9qjwfWlDU1uJkm7nmlLVHnmlqgQ/cKNwqPvijNJjZMrUh61GDQckiq3ESZo3cUzNJmkwbHkbjwaAjetMyc8UZJ70guc6Ce9KKj3GlDGsbEku6lBzTEUsatRxhRz1qooBI4iTk1aTCjApgOKUGtUholzmlHvUeaN3FAiTIpM1GWpVNJuwbEgNBNMVqUmi4XHZoBpueevFANFwH5opuaCRTAdmjNNzRmi9xjqQnimluOtV5pscUXsA6WbAODVZ3LGkY5pBWTdwAHmnCm5oz6UkFyRTzQTzTN2KN1VdFDieKTrTc0ZosMeuAfSgmmHNHIotYkcKUdaZmlBouMdml3Un1pKVgHbqcpqPFPj5NCigLEZwRVjfxUCjin5yPStUhjsndkE08Ng1Eo455pW6cUcomS76crjcu4ZXIz9KqlzSbyR1osBv6Ns+13EZUpKo+VkJJODyOveobuFY5bnZEu2NyBksBxnqc4z2qKyJE8d0soQlMEbsFjgg/wAqtPM8lq1t5oMJ3Arn1xj8jzW6inERQV0e3djGqMCoGCxPP1PoKYGpLkeTHHCTk5ZiR35wP5VGDkVi9GFibNITioc0E1ImiZWzTs1WDEGjf70gMsRg9qkSAHtinxpiphVqAhqRqBwKk2imE0ZrTlSGOKe9AX1pA1OJoskAu2m4pC1KKVhCEHNLg08AUuKlxBjMEUoBpxNJuFKwDSKTNPJpBijlATmgnmlJBxQABQlcBM1PNCHjD23zeqk8iq7HmopZCB8pIPqDyKbVikxJRKeAkgx1OwnFUrHT9fLStfaZdrGzfum8o/MvrXoHg3xdqatBpuIZFVSI3aMZ+hPf610cfjS/TT9YlGfNsiIirEMpz0Irqp4ZShzMqx5I9vcI+14ZEbvvUrj86RiiDaQWY/xKeBV7W9YvNcvBd6lL5soQIoAwqj2FZ52nt0rimlF2QaIbkHrRkUhXJpdvFZWJYbqAQabtpdtFhLQeDSk+lNANB4oQhdxFAYmkBpM0FJik09etMp/amFwZqQdaaetKKEwQ/tUkA5yajXk1ajAwM4q4lEgPFAxjrTSR2NIDViJAQKGYYqItimF6AHmgdaYDS5oGXbe42QlDlSCSrAA9evFTfaSrh3niYgg/u48k/ngCs7PFL2q+awrD55fNkLgFVxgD0AFCN71ATSg4NSwuWMj1oJ9KiEmBTRJk0rATZoAB61CX96UOaVgP/9k='}
def demo(which=None):
    items = _DEMO_B64.items() if which is None else [(which, _DEMO_B64[which])]
    for name, b64 in items:
        safe_judge(base64.b64decode(b64), label=f'[demo] {name}')
print('demo() ready — bundled examples:', list(_DEMO_B64))

## 4. Upload images and get verdicts
Run the cell, pick one or more images from your computer (each is one API call).
Keep them reasonably sized — phone photos are fine.

In [ ]:
from google.colab import files
uploaded = files.upload()
for name, data in uploaded.items():
    safe_judge(data, label=name)

## 5. Or judge an image from a URL (https only)

In [ ]:
import urllib.request
IMAGE_URL = ''  #@param {type:"string"}
if IMAGE_URL.startswith('https://'):
    with urllib.request.urlopen(IMAGE_URL, timeout=30) as r:
        safe_judge(r.read(), label=IMAGE_URL.rsplit('/', 1)[-1][:60])
else:
    print('paste an https:// image URL into IMAGE_URL first')

## 6. Upload a VIDEO (.mp4) — every frame analyzed, itemized report
**Pass 1:** every frame goes through the contact + tackle classifiers
**locally — 0 API calls.** **Pass 2:** the clip is split into segments, and
only the segments that *flag contact* get the full image analysis (detector
+ ball). Quiet stretches cost nothing.

You get an **itemized report**: a timeline of every segment, a detailed
verdict for each contact event, and a one-line summary.

**Run cells 1–3 first.** `judge_video(data, at_seconds=X)` forces analysis
onto a specific moment. Longer clips are evenly sampled to `max_frames=500`.

In [ ]:
from google.colab import files
from airef.video_judge import judge_video

def show_video_result(name, data):
    print(f'--- {name} ---')
    v = judge_video(data)
    ts = [t for t, _ in v.timeline]; ss = [s for _, s in v.timeline]
    plt.figure(figsize=(9, 3)); plt.plot(ts, ss, lw=1)
    plt.axhline(0.55, c='gray', ls='--', label='contact bar')
    if v.is_foul:
        plt.axvline(v.peak_t, c='red', ls=':', label=f'event ~{v.peak_t:.1f}s')
    plt.xlabel('seconds'); plt.ylabel('P(contact) — every frame, local'); plt.ylim(0, 1.05)
    plt.legend(); plt.grid(alpha=0.3); plt.title('per-frame contact probability (all frames)'); plt.show()
    img = cv2.cvtColor(v.peak_frame, cv2.COLOR_BGR2RGB)
    taken = []
    for d in sorted(v.peak_detections, key=lambda d: -d.confidence):
        x0, y0 = int(d.x - d.width / 2), int(d.y - d.height / 2)
        x1, y1 = int(d.x + d.width / 2), int(d.y + d.height / 2)
        color = (255, 60, 60) if d.cls in FOUL_CLASSES else (60, 120, 255)
        cv2.rectangle(img, (x0, y0), (x1, y1), color, 2)
        _draw_label(img, f'{d.cls} {d.confidence:.2f}', x0, y0, color, taken)
    plt.figure(figsize=(8, 6)); plt.imshow(img); plt.axis('off')
    plt.title(f'best frame @ {v.peak_t:.1f}s'); plt.show()
    print('=' * 56); print(v.report); print('=' * 56)

up = files.upload()
for name, data in up.items():
    try:
        show_video_result(name, data)
    except Exception as e:
        print(f'\u26a0 could not analyze {name}: {type(e).__name__}: {e}')
        print('  checklist: internet on? cell 1 API key? cell 2b weights loaded?')